# Recomendación de planes móviles para Megaline

## Descripción del proyecto

El operador móvil **Megaline** quiere que sus clientes con planes heredados se cambien a uno de sus planes actuales: **Smart** o **Ultra**.

Se construye un modelo de clasificación que, a partir del comportamiento mensual de cada cliente, recomienda el plan adecuado. La meta es alcanzar una exactitud (accuracy) de al menos **0.75**.

## Datos

`users_behavior.csv`, con un registro por cliente y mes:

- `calls`: número de llamadas.
- `minutes`: duración total de las llamadas en minutos.
- `messages`: número de mensajes de texto.
- `mb_used`: tráfico de internet en MB.
- `is_ultra`: plan del cliente (1 = Ultra, 0 = Smart).

## Pasos

1. Cargar los datos y separar características y objetivo.
2. Dividir en entrenamiento, validación y prueba (60/20/20).
3. Comparar árbol de decisión, bosque aleatorio y regresión logística, ajustando hiperparámetros con el conjunto de validación.
4. Evaluar en el conjunto de prueba y realizar una prueba de cordura.

**Herramientas:** Python, pandas, scikit-learn.

# Importar librerías

In [32]:

#Librerías
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import accuracy_score

# 1. Cargue de datos

In [25]:

# Cargue de datos
df = pd.read_csv('/datasets/users_behavior.csv')
df.head()


,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


# 2. Segmentación de datos

In [26]:
features = df.drop('is_ultra', axis=1)
target = df['is_ultra']

In [27]:
# Primera división: 80% entrenamiento+validación, 20% prueba
features_train_valid, features_test, target_train_valid, target_test = train_test_split(
    features,
    target,
    test_size=0.20,
    random_state=12345
)

# Segunda división: del 80% restante, 25% para validación
# 25% de 80% = 20% del total
features_train, features_valid, target_train, target_valid = train_test_split(
    features_train_valid,
    target_train_valid,
    test_size=0.25,
    random_state=12345
)

# 3. Calidad de modelos - 4. Calidad en conjunto de prueba

Ya que la variable objetivo 'is_ultra' consta de "1" o "0" se realizará la validacion con modelos de clasificación.

## Árbol de decisión en diferentes profundidades

### Calidad modelo con diferentes hiperparámetros

In [28]:
best_result = 0
best_model = None
best_depth = None

for depth in range(1, 21): 
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=12345
    )
    
    model.fit(features_train, target_train)
    
    predictions_valid = model.predict(features_valid)
    
    result = accuracy_score(target_valid, predictions_valid)
    
    if result > best_result:
        best_model = model
        best_result = result
        best_depth = depth

print(
    f"Accuracy del mejor modelo en el conjunto de validación "
    f"(max_depth = {best_depth}): {best_result:.4f}"
)

Accuracy del mejor modelo en el conjunto de validación (max_depth = 7): 0.7745


### Calidad del modelo en conjunto de prueba

In [29]:
predictions_test = best_model.predict(features_test)

test_accuracy = accuracy_score(target_test, predictions_test)

print(
    f"Accuracy del mejor modelo en el conjunto de prueba: "
    f"{test_accuracy:.4f}"
)

Accuracy del mejor modelo en el conjunto de prueba: 0.7885


### Conclusión

Se investigó la calidad de un modelo de árbol de decisión modificando el hiperparámetro `max_depth` con valores entre 1 y 20. Para seleccionar el mejor modelo se utilizó el conjunto de validación.

El mejor resultado se obtuvo con una profundidad máxima de **7**, alcanzando una exactitud (`accuracy`) de **0.7745**, es decir, aproximadamente el **77.45 %** de las predicciones fueron correctas en el conjunto de validación.

Posteriormente, se comprobó la calidad del modelo seleccionado utilizando el conjunto de prueba, que no se había utilizado para elegir el hiperparámetro. El modelo obtuvo una exactitud de **0.7885**, equivalente al **78.85 %** de predicciones correctas.

Los resultados muestran que el modelo seleccionado mantiene un rendimiento similar e incluso ligeramente superior en el conjunto de prueba respecto al conjunto de validación, por lo que no se observa una diferencia importante entre ambos resultados.



## Bosque aleatorio en diferentes profundidades y cantidades de árboles

### Calidad modelo con diferentes hiperparámetros

In [30]:
best_result = 0
best_model = None
best_estimators = 0
best_depth = 0

for est in range (10, 101, 10):
    for depth in range (1,11):
        model = RandomForestClassifier(
            random_state=12345,
            n_estimators = est,
            max_depth = depth
        )

        model.fit(features_train, target_train)
        predictions_valid = model.predict(features_valid)

        result = accuracy_score(target_valid, predictions_valid)
    
        if result > best_result:
            best_model = model
            best_result = result
            best_estimators = est
            best_depth = depth

print(
    f"Accuracy del mejor modelo en el conjunto de validación "
    f"(n_estimators = {best_estimators}, "
    f"max_depth = {best_depth}): {best_result:.4f}"
)

Accuracy del mejor modelo en el conjunto de validación (n_estimators = 50, max_depth = 10): 0.7978


### Calidad del modelo en conjunto de prueba

In [31]:
predictions_test = best_model.predict(features_test)

test_accuracy = accuracy_score(
    target_test,
    predictions_test
)

print(
    f"Accuracy del mejor modelo en el conjunto de prueba: "
    f"{test_accuracy:.4f}"
)

Accuracy del mejor modelo en el conjunto de prueba: 0.7994


### Conclusión

Se investigó la calidad de un modelo de Bosque Aleatorio modificando dos hiperparámetros: n_estimators, con valores entre 10 y 100, y max_depth, con valores entre 1 y 10. En total se evaluaron diferentes combinaciones de estos parámetros utilizando el conjunto de validación.

La mejor combinación fue n_estimators = 50 y max_depth = 10, con una exactitud (accuracy) de 0.7978, equivalente al 79.78 % de predicciones correctas en el conjunto de validación.

Posteriormente, se evaluó el mejor modelo utilizando el conjunto de prueba. El modelo obtuvo una exactitud de 0.7994, es decir, un 79.94 % de predicciones correctas.

La diferencia entre el resultado de validación y el de prueba es de aproximadamente 0.16 puntos porcentuales, por lo que el rendimiento del modelo se mantiene muy similar en ambos conjuntos.




## Regresión logística

### Calidad modelo

In [36]:
model = LogisticRegression(
    max_iter=1000,
    random_state=12345
)

model.fit(features_train, target_train)

predictions_valid = model.predict(features_valid)

valid_accuracy = accuracy_score(
    target_valid,
    predictions_valid
)

print(
    f"Accuracy de la regresión logística en validación: "
    f"{valid_accuracy:.4f}"
)

Accuracy de la regresión logística en validación: 0.7263


### Calidad del modelo en conjunto de prueba

In [37]:
predictions_test = model.predict(features_test)

test_accuracy = accuracy_score(
    target_test,
    predictions_test
)

print(
    f"Accuracy de la regresión logística en prueba: "
    f"{test_accuracy:.4f}"
)

Accuracy de la regresión logística en prueba: 0.7589


### Conclusión

La regresión logística obtuvo una accuracy de **0.7263 (72.63 %)** en el conjunto de validación y de **0.7589 (75.89 %)** en el conjunto de prueba. Esto indica que el modelo mantuvo un rendimiento relativamente estable al evaluarse con datos nuevos, aunque su precisión en el conjunto de prueba fue ligeramente superior a la obtenida en validación.


### Conclusión final

Se evaluaron tres modelos de clasificación para predecir la variable `is_ultra`: árbol de decisión, bosque aleatorio y regresión logística.

Para el árbol de decisión se probaron diferentes valores de `max_depth`, entre 1 y 20. El mejor resultado se obtuvo con `max_depth = 7`, alcanzando una accuracy de **0.7745** en validación y **0.7885** en el conjunto de prueba.

Para el bosque aleatorio se probaron diferentes combinaciones de `n_estimators` y `max_depth`. La mejor configuración fue `n_estimators = 50` y `max_depth = 10`, con una accuracy de **0.7978** en validación y **0.7994** en el conjunto de prueba.

Finalmente, se evaluó una regresión logística sin modificar sus hiperparámetros. Este modelo obtuvo una accuracy de **0.7263** en validación y **0.7589** en el conjunto de prueba.

En general, los resultados muestran que los modelos basados en árboles obtuvieron una mayor accuracy que la regresión logística en este conjunto de datos. El bosque aleatorio alcanzó una accuracy de **0.7994** en el conjunto de prueba. La diferencia entre los resultados de validación y prueba fue pequeña en los tres modelos, lo que indica que su rendimiento se mantuvo relativamente estable al evaluarlos con datos que no habían sido utilizados durante el entrenamiento.


# 5. Prueba de cordura

Se realiza una prueba respecto del modelo con mayor accuracy en prueba (Bosque Aleatorio)

In [38]:
# Seleccionamos algunos ejemplos del conjunto de prueba
test_examples = features_test.head(10)

# Predicciones del modelo
predictions = best_model.predict(test_examples)

# Mostramos las predicciones
print("Predicciones del modelo:")
print(predictions)

# Mostramos los valores reales
print("\nValores reales:")
print(target_test.head(10).values)

Predicciones del modelo:
[0 0 0 0 0 0 0 1 0 0]

Valores reales:
[1 0 0 1 0 0 0 0 0 0]
